In [5]:
# 
# ANORTHITE DYNAMICS — FAST A/B/C STATE REPRESENTATION EXPERIMENT
#
# A = species
# B = species + velocity
# C = species + velocity + force
#
# TRAIN: seed 11001 — 6000 samples
# VAL:   seed 22002 — 6000 samples
# TEST:  PHYSICALLY ABSENT / NEVER LOADED
#
# Optimized for time-metered Colab:
#   - NPZ files loaded ONCE into RAM
#   - train normalization computed during cache load
#   - cached tensors reused for A/B/C
#   - mixed precision on CUDA
#   - validation every 5 epochs
#   - larger but conservative batch size for T4
# 

%pip -q install torch-geometric

import os
import json
import random
import time

import numpy as np
import torch
import torch.nn as nn

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import NNConv


# 
# Configuration
# 

ROOT = "/content/processed_production"
SPLIT_FILE = os.path.join(ROOT, "splits.json")
OUTPUT_DIR = "/content/abc_outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

SEED = 318

# 48 is substantially faster than 32 while staying conservative
# for NNConv's large per-edge weight matrices on a T4.
BATCH_SIZE = 48

EPOCHS = 200
LR = 1e-3
WEIGHT_DECAY = 0.0
HIDDEN = 64
CUTOFF = 3.5

# Full validation is expensive because VAL = 6000 graphs.
VAL_EVERY = 5

# 6 validation checks = 30 epochs without improvement,
# equivalent in spirit to the old patience=30.
PATIENCE_EVALS = 6

# FP16/AMP saves substantial T4 time.
USE_AMP = DEVICE.type == "cuda"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Mixed precision:", USE_AMP)


# 
# Read official split manifest
# 

with open(SPLIT_FILE, "r") as f:
    split_info = json.load(f)

train_runs = split_info["splits"]["train"]
val_runs = split_info["splits"]["val"]
test_runs = split_info["splits"]["test"]

assert len(train_runs) == 12
assert len(val_runs) == 12
assert len(test_runs) == 12

print()
print("Train trajectories:", len(train_runs))
print("Val trajectories:  ", len(val_runs))
print("Test trajectories: ", len(test_runs), "(manifest only)")


# 
# Verify test data is physically absent
# 

test_present = []

for entry in test_runs:
    path = os.path.join(ROOT, entry["run_id"])

    if os.path.exists(path):
        test_present.append(entry["run_id"])

if test_present:
    raise RuntimeError(
        "TEST DATA FOUND IN RUNTIME:\n"
        + "\n".join(test_present)
    )

print("Physical test trajectories present: 0")
print("TEST SET IS SEALED.")


# 
# Resolve sample paths
# 

def get_sample_paths(entries):

    paths = []

    for entry in entries:

        # splits.json was created on Windows.
        # Convert Windows separators to Linux/Colab separators.
        relative_sample_dir = (
            entry["sample_dir"]
            .replace("\\", "/")
        )

        sample_dir = os.path.join(
            ROOT,
            relative_sample_dir
        )

        if not os.path.isdir(sample_dir):
            raise FileNotFoundError(
                f"{entry['run_id']}: sample directory not found:\n"
                f"{sample_dir}"
            )

        files = sorted(
            os.path.join(sample_dir, f)
            for f in os.listdir(sample_dir)
            if f.startswith("sample_")
            and f.endswith(".npz")
        )

        expected = entry["num_samples"]

        if len(files) != expected:
            raise RuntimeError(
                f"{entry['run_id']}: "
                f"expected {expected} samples, "
                f"got {len(files)}"
            )

        paths.extend(files)

    return paths


train_paths = get_sample_paths(train_runs)
val_paths = get_sample_paths(val_runs)

assert len(train_paths) == 6000
assert len(val_paths) == 6000

print()
print("Train samples:", len(train_paths))
print("Val samples:  ", len(val_paths))
print()
print("First train sample:", train_paths[0])
print("First val sample:  ", val_paths[0])

assert os.path.isfile(train_paths[0])
assert os.path.isfile(val_paths[0])

print("PATH CHECK PASSED.")


train_paths = get_sample_paths(train_runs)
val_paths = get_sample_paths(val_runs)

assert len(train_paths) == 6000
assert len(val_paths) == 6000

print()
print("Train samples:", len(train_paths))
print("Val samples:  ", len(val_paths))


# 
# Base graph container
#
# Everything gets loaded exactly once from disk.
# 

class CachedGraph:

    __slots__ = (
        "species",
        "velocity",
        "force",
        "edge_index",
        "edge_attr",
        "y"
    )

    def __init__(
        self,
        species,
        velocity,
        force,
        edge_index,
        edge_attr,
        y
    ):
        self.species = species
        self.velocity = velocity
        self.force = force
        self.edge_index = edge_index
        self.edge_attr = edge_attr
        self.y = y


# 
# Load TRAIN once + accumulate normalization statistics
# 

def load_train_cache(paths):

    cache = []

    vel_sum = np.zeros(3, dtype=np.float64)
    vel_sq_sum = np.zeros(3, dtype=np.float64)

    force_sum = np.zeros(3, dtype=np.float64)
    force_sq_sum = np.zeros(3, dtype=np.float64)

    n_atoms = 0

    start = time.time()

    for i, path in enumerate(paths):

        with np.load(path) as d:

            species = (
                d["atom_types"]
                .astype(np.float32)[:, None]
            )

            velocity = np.stack(
                [
                    d["vx_t"],
                    d["vy_t"],
                    d["vz_t"]
                ],
                axis=1
            ).astype(np.float32)

            force = np.stack(
                [
                    d["fx_t"],
                    d["fy_t"],
                    d["fz_t"]
                ],
                axis=1
            ).astype(np.float32)

            src = d["edge_src"].astype(np.int64)
            dst = d["edge_dst"].astype(np.int64)

            edge_index = np.stack(
                [src, dst],
                axis=0
            )

            edge_vector = (
                d["edge_vector"]
                .astype(np.float32)
            )

            edge_distance = (
                d["edge_distance"]
                .astype(np.float32)
            )

            direction = edge_vector / np.maximum(
                edge_distance[:, None],
                1e-8
            )

            edge_attr = np.concatenate(
                [
                    direction,
                    edge_distance[:, None] / CUTOFF
                ],
                axis=1
            ).astype(np.float32)

            y = (
                d["non_affine_displacement"]
                .astype(np.float32)
            )

        # Train-only statistics.
        vel64 = velocity.astype(np.float64)
        force64 = force.astype(np.float64)

        vel_sum += vel64.sum(axis=0)
        vel_sq_sum += (vel64 ** 2).sum(axis=0)

        force_sum += force64.sum(axis=0)
        force_sq_sum += (force64 ** 2).sum(axis=0)

        n_atoms += velocity.shape[0]

        cache.append(
            CachedGraph(
                torch.from_numpy(species),
                torch.from_numpy(velocity),
                torch.from_numpy(force),
                torch.from_numpy(edge_index),
                torch.from_numpy(edge_attr),
                torch.from_numpy(y)
            )
        )

        if (i + 1) % 1000 == 0:
            print(
                f"Train cache: "
                f"{i+1}/{len(paths)}"
            )

    vel_mean = vel_sum / n_atoms
    force_mean = force_sum / n_atoms

    vel_var = (
        vel_sq_sum / n_atoms
        - vel_mean ** 2
    )

    force_var = (
        force_sq_sum / n_atoms
        - force_mean ** 2
    )

    vel_std = np.sqrt(
        np.maximum(vel_var, 1e-12)
    )

    force_std = np.sqrt(
        np.maximum(force_var, 1e-12)
    )

    norm = {
        "velocity_mean":
            torch.tensor(
                vel_mean,
                dtype=torch.float32
            ),

        "velocity_std":
            torch.tensor(
                vel_std,
                dtype=torch.float32
            ),

        "force_mean":
            torch.tensor(
                force_mean,
                dtype=torch.float32
            ),

        "force_std":
            torch.tensor(
                force_std,
                dtype=torch.float32
            )
    }

    print(
        f"Train cache loaded in "
        f"{time.time()-start:.1f} s"
    )

    return cache, norm


train_cache, norm = load_train_cache(
    train_paths
)


# 
# Normalize train cache ONCE
# 

for g in train_cache:

    g.velocity = (
        g.velocity
        - norm["velocity_mean"]
    ) / norm["velocity_std"]

    g.force = (
        g.force
        - norm["force_mean"]
    ) / norm["force_std"]


print()
print("TRAIN-ONLY NORMALIZATION")

print(
    "Velocity mean:",
    norm["velocity_mean"].numpy()
)

print(
    "Velocity std: ",
    norm["velocity_std"].numpy()
)

print(
    "Force mean:   ",
    norm["force_mean"].numpy()
)

print(
    "Force std:    ",
    norm["force_std"].numpy()
)


# 
# Load VAL once
#
# Uses TRAIN normalization.
# 

def load_val_cache(paths, norm):

    cache = []

    start = time.time()

    for i, path in enumerate(paths):

        with np.load(path) as d:

            species = (
                d["atom_types"]
                .astype(np.float32)[:, None]
            )

            velocity = np.stack(
                [
                    d["vx_t"],
                    d["vy_t"],
                    d["vz_t"]
                ],
                axis=1
            ).astype(np.float32)

            force = np.stack(
                [
                    d["fx_t"],
                    d["fy_t"],
                    d["fz_t"]
                ],
                axis=1
            ).astype(np.float32)

            src = d["edge_src"].astype(np.int64)
            dst = d["edge_dst"].astype(np.int64)

            edge_index = np.stack(
                [src, dst],
                axis=0
            )

            edge_vector = (
                d["edge_vector"]
                .astype(np.float32)
            )

            edge_distance = (
                d["edge_distance"]
                .astype(np.float32)
            )

            direction = edge_vector / np.maximum(
                edge_distance[:, None],
                1e-8
            )

            edge_attr = np.concatenate(
                [
                    direction,
                    edge_distance[:, None] / CUTOFF
                ],
                axis=1
            ).astype(np.float32)

            y = (
                d["non_affine_displacement"]
                .astype(np.float32)
            )

        velocity = (
            torch.from_numpy(velocity)
            - norm["velocity_mean"]
        ) / norm["velocity_std"]

        force = (
            torch.from_numpy(force)
            - norm["force_mean"]
        ) / norm["force_std"]

        cache.append(
            CachedGraph(
                torch.from_numpy(species),
                velocity,
                force,
                torch.from_numpy(edge_index),
                torch.from_numpy(edge_attr),
                torch.from_numpy(y)
            )
        )

        if (i + 1) % 1000 == 0:
            print(
                f"Val cache: "
                f"{i+1}/{len(paths)}"
            )

    print(
        f"Val cache loaded in "
        f"{time.time()-start:.1f} s"
    )

    return cache


val_cache = load_val_cache(
    val_paths,
    norm
)

print()
print(
    "All 12,000 train+val graphs "
    "are now cached in RAM."
)


# 
# Useful target-scale baseline
#
# Zero predictor:
# predict zero non-affine displacement.
# 

def zero_baseline(cache):

    squared = 0.0
    absolute = 0.0
    n = 0

    for g in cache:

        squared += torch.sum(
            g.y ** 2
        ).item()

        absolute += torch.sum(
            torch.abs(g.y)
        ).item()

        n += g.y.numel()

    mse = squared / n

    return np.sqrt(mse), absolute / n


zero_rmse, zero_mae = zero_baseline(
    val_cache
)

print()
print("VAL ZERO-PREDICTOR BASELINE")
print(f"RMSE: {zero_rmse:.8f} Å")
print(f"MAE:  {zero_mae:.8f} Å")


# 
# Representation view over cached graphs
# 

class CachedAtomisticDataset(
    torch.utils.data.Dataset
):

    def __init__(self, cache, variant):

        assert variant in ["A", "B", "C"]

        self.cache = cache
        self.variant = variant

    def __len__(self):
        return len(self.cache)

    def __getitem__(self, idx):

        g = self.cache[idx]

        if self.variant == "A":

            x = g.species

        elif self.variant == "B":

            x = torch.cat(
                [
                    g.species,
                    g.velocity
                ],
                dim=1
            )

        else:

            x = torch.cat(
                [
                    g.species,
                    g.velocity,
                    g.force
                ],
                dim=1
            )

        return Data(
            x=x,
            edge_index=g.edge_index,
            edge_attr=g.edge_attr,
            y=g.y
        )


# 
# Model
# 

class DynamicsGNN(nn.Module):

    def __init__(self, in_dim):

        super().__init__()

        self.input = nn.Linear(
            in_dim,
            HIDDEN
        )

        edge_net1 = nn.Sequential(
            nn.Linear(4, 64),
            nn.SiLU(),
            nn.Linear(
                64,
                HIDDEN * HIDDEN
            )
        )

        self.conv1 = NNConv(
            HIDDEN,
            HIDDEN,
            edge_net1,
            aggr="mean"
        )

        edge_net2 = nn.Sequential(
            nn.Linear(4, 64),
            nn.SiLU(),
            nn.Linear(
                64,
                HIDDEN * HIDDEN
            )
        )

        self.conv2 = NNConv(
            HIDDEN,
            HIDDEN,
            edge_net2,
            aggr="mean"
        )

        self.head = nn.Sequential(
            nn.Linear(
                HIDDEN,
                HIDDEN
            ),
            nn.SiLU(),
            nn.Linear(
                HIDDEN,
                3
            )
        )

    def forward(self, data):

        x = self.input(data.x)

        x = torch.nn.functional.silu(x)

        residual = x

        x = self.conv1(
            x,
            data.edge_index,
            data.edge_attr
        )

        x = torch.nn.functional.silu(x)

        x = self.conv2(
            x,
            data.edge_index,
            data.edge_attr
        )

        x = torch.nn.functional.silu(
            x + residual
        )

        return self.head(x)


# 
# Evaluation
# 

@torch.no_grad()
def evaluate(model, loader):

    model.eval()

    sum_squared = 0.0
    sum_absolute = 0.0
    n_values = 0

    for batch in loader:

        batch = batch.to(
            DEVICE,
            non_blocking=True
        )

        with torch.amp.autocast(
            device_type="cuda",
            enabled=USE_AMP
        ):
            pred = model(batch)

        # Calculate reported metrics in FP32.
        error = (
            pred.float()
            - batch.y.float()
        )

        sum_squared += torch.sum(
            error ** 2
        ).item()

        sum_absolute += torch.sum(
            torch.abs(error)
        ).item()

        n_values += error.numel()

    mse = sum_squared / n_values

    return (
        mse,
        np.sqrt(mse),
        sum_absolute / n_values
    )


# 
# Train one representation
# 

def train_variant(variant):

    print()
    print("=" * 72)
    print(f"VARIANT {variant}")
    print("=" * 72)

    # Same starting seed for every ablation.
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    train_dataset = CachedAtomisticDataset(
        train_cache,
        variant
    )

    val_dataset = CachedAtomisticDataset(
        val_cache,
        variant
    )

    # Explicit generator gives each variant the same
    # deterministic shuffle sequence.
    generator = torch.Generator()
    generator.manual_seed(SEED)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=DEVICE.type == "cuda"
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=DEVICE.type == "cuda"
    )

    first = train_dataset[0]

    print(
        "Node feature dimension:",
        first.x.shape[1]
    )

    print(
        "Edge feature dimension:",
        first.edge_attr.shape[1]
    )

    print(
        "Target dimension:",
        first.y.shape[1]
    )

    model = DynamicsGNN(
        first.x.shape[1]
    ).to(DEVICE)

    parameter_count = sum(
        p.numel()
        for p in model.parameters()
    )

    print(
        "Parameters:",
        parameter_count
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    criterion = nn.MSELoss()

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP
    )

    best_val = float("inf")
    best_epoch = 0

    evals_without_improvement = 0

    checkpoint = os.path.join(
        OUTPUT_DIR,
        f"best_variant_{variant}.pt"
    )

    history = []

    variant_start = time.time()

    for epoch in range(
        1,
        EPOCHS + 1
    ):

        epoch_start = time.time()

        model.train()

        train_squared = 0.0
        train_values = 0

        for batch in train_loader:

            batch = batch.to(
                DEVICE,
                non_blocking=True
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            with torch.amp.autocast(
                device_type="cuda",
                enabled=USE_AMP
            ):

                pred = model(batch)

                loss = criterion(
                    pred,
                    batch.y
                )

            scaler.scale(
                loss
            ).backward()

            scaler.step(
                optimizer
            )

            scaler.update()

            error = (
                pred.detach().float()
                - batch.y.float()
            )

            train_squared += torch.sum(
                error ** 2
            ).item()

            train_values += error.numel()

        train_mse = (
            train_squared
            / train_values
        )

        train_rmse = np.sqrt(
            train_mse
        )

        # --------------------------------------------
        # Only perform expensive full validation when
        # necessary.
        # --------------------------------------------

        do_validation = (
            epoch == 1
            or epoch % VAL_EVERY == 0
            or epoch == EPOCHS
        )

        val_mse = None
        val_rmse = None
        val_mae = None

        if do_validation:

            (
                val_mse,
                val_rmse,
                val_mae
            ) = evaluate(
                model,
                val_loader
            )

            if val_mse < best_val:

                best_val = val_mse
                best_epoch = epoch

                evals_without_improvement = 0

                torch.save(
                    model.state_dict(),
                    checkpoint
                )

            else:

                evals_without_improvement += 1

        history.append({
            "epoch": epoch,
            "train_rmse":
                float(train_rmse),
            "val_rmse":
                None
                if val_rmse is None
                else float(val_rmse),
            "val_mae":
                None
                if val_mae is None
                else float(val_mae)
        })

        if do_validation:

            print(
                f"Epoch {epoch:03d} | "
                f"train RMSE "
                f"{train_rmse:.6e} Å | "
                f"val RMSE "
                f"{val_rmse:.6e} Å | "
                f"val MAE "
                f"{val_mae:.6e} Å | "
                f"{time.time()-epoch_start:.1f}s"
            )

        if (
            do_validation
            and
            evals_without_improvement
            >= PATIENCE_EVALS
        ):

            print(
                f"Early stopping at "
                f"epoch {epoch}. "
                f"Best epoch: "
                f"{best_epoch}"
            )

            break

    # ========================================================
    # Reload exact best validation checkpoint
    # ========================================================

    model.load_state_dict(
        torch.load(
            checkpoint,
            map_location=DEVICE,
            weights_only=True
        )
    )

    (
        val_mse,
        val_rmse,
        val_mae
    ) = evaluate(
        model,
        val_loader
    )

    elapsed = (
        time.time()
        - variant_start
    )

    print()
    print(
        f"BEST VARIANT {variant}"
    )

    print(
        "Best epoch:",
        best_epoch
    )

    print(
        f"Validation RMSE: "
        f"{val_rmse:.8f} Å"
    )

    print(
        f"Validation MAE:  "
        f"{val_mae:.8f} Å"
    )

    print(
        f"Wall time: "
        f"{elapsed/60:.2f} min"
    )

    with open(
        os.path.join(
            OUTPUT_DIR,
            f"history_variant_{variant}.json"
        ),
        "w"
    ) as f:

        json.dump(
            history,
            f,
            indent=2
        )

    # Free GPU memory before next representation.
    del model
    del optimizer

    torch.cuda.empty_cache()

    return {
        "variant": variant,
        "best_epoch": best_epoch,
        "val_mse": float(val_mse),
        "val_rmse": float(val_rmse),
        "val_mae": float(val_mae),
        "parameters": parameter_count,
        "wall_seconds": float(elapsed),
        "checkpoint": checkpoint
    }


# 
# Run A/B/C
# 

experiment_start = time.time()

results = []

for variant in [
    "A",
    "B",
    "C"
]:

    results.append(
        train_variant(
            variant
        )
    )


# 
# Final validation comparison
#
# TEST REMAINS PHYSICALLY ABSENT.
# 

print()
print("=" * 72)
print("FINAL VALIDATION COMPARISON")
print("=" * 72)

print()
print(
    f"Zero baseline | "
    f"RMSE {zero_rmse:.8f} Å | "
    f"MAE {zero_mae:.8f} Å"
)

for r in results:

    improvement = (
        1.0
        - r["val_rmse"]
        / zero_rmse
    ) * 100.0

    print(
        f"{r['variant']} | "
        f"epoch {r['best_epoch']:3d} | "
        f"RMSE "
        f"{r['val_rmse']:.8f} Å | "
        f"MAE "
        f"{r['val_mae']:.8f} Å | "
        f"vs-zero RMSE "
        f"{improvement:+.2f}%"
    )

best = min(
    results,
    key=lambda x: x["val_rmse"]
)

total_minutes = (
    time.time()
    - experiment_start
) / 60

print()
print(
    "Best validation representation:",
    best["variant"]
)

print(
    f"Total A/B/C training time: "
    f"{total_minutes:.2f} min"
)

print()
print(
    "TEST SET HAS NOT BEEN "
    "LOADED OR EVALUATED."
)

with open(
    os.path.join(
        OUTPUT_DIR,
        "abc_results.json"
    ),
    "w"
) as f:

    json.dump(
        {
            "seed": SEED,
            "batch_size": BATCH_SIZE,
            "val_every": VAL_EVERY,
            "zero_baseline_rmse":
                float(zero_rmse),
            "zero_baseline_mae":
                float(zero_mae),
            "results": results
        },
        f,
        indent=2
    )

Device: cuda


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\games\\Downloads\\Diopside_Anorthite_MatsuiPot\\Diopside_Anorthite_MatsuiPot\\processed_production/splits.json'

In [ ]:
# ============================================================
# ANORTHITE DYNAMICS
# E(3)-EQUIVARIANT A/B/C STATE REPRESENTATION EXPERIMENT
# + AUTOMATIC SO(3) ROTATION STRESS TEST
#
# Environment verified:
#   PyTorch 2.11.0
#   e3nn 0.6.0
#
# REPRESENTATIONS
# 
# A = species
# B = species + velocity
# C = species + velocity + force
#
# E(3) interpretation:
#
#   species  -> scalar 0e
#   velocity -> vector 1o
#   force    -> vector 1o
#   edge r   -> spherical harmonics
#   target   -> vector 1o
#
# TRAIN: seed 11001 — 6000 samples
# VAL:   seed 22002 — 6000 samples
# TEST:  PHYSICALLY ABSENT / NEVER LOADED
#
# IMPORTANT:
#   - train-only normalization
#   - same trajectory split as NNConv
#   - cached RAM dataset
#   - best checkpoint chosen ONLY by validation MSE
#   - no test-set access
#   - no rotation augmentation
#   - rotation test occurs AFTER training
# ============================================================


# ============================================================
# Install
# ============================================================

%pip -q install e3nn==0.6.0 torch-geometric


# ============================================================
# Imports
# ============================================================

import os
import json
import math
import random
import time
import shutil

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

from e3nn import o3


# ============================================================
# Configuration
# ============================================================

ROOT = "/content/processed_production"

SPLIT_FILE = os.path.join(
    ROOT,
    "splits.json"
)

OUTPUT_DIR = "/content/e3nn_abc_outputs"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

SEED = 318
ROTATION_SEED = 8675309

BATCH_SIZE = 48

EPOCHS = 200

LR = 1e-3
WEIGHT_DECAY = 0.0

CUTOFF = 3.5

VAL_EVERY = 5
PATIENCE_EVALS = 6

N_ROTATIONS = 10

# 
# Equivariant model size
#
# Hidden irreps contain:
#
#   32 scalar-even channels
#   16 vector-odd channels
#
# This is intentionally modest for a T4.
# 

HIDDEN_IRREPS = o3.Irreps(
    "32x0e + 16x1o"
)

# Edge spherical harmonics up through l=2.
EDGE_IRREPS = o3.Irreps.spherical_harmonics(
    lmax=2
)

# Output is a polar vector.
OUTPUT_IRREPS = o3.Irreps(
    "1x1o"
)

# Number of radial basis functions used to generate
# tensor-product weights.
NUM_RADIAL = 8

RADIAL_HIDDEN = 64


# ============================================================
# Reproducibility
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        SEED
    )

print("Device:", DEVICE)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

print(
    "PyTorch:",
    torch.__version__
)

import e3nn

print(
    "e3nn:",
    e3nn.__version__
)

print(
    "Hidden irreps:",
    HIDDEN_IRREPS
)

print(
    "Edge irreps:",
    EDGE_IRREPS
)

print(
    "Output irreps:",
    OUTPUT_IRREPS
)

print()
print(
    "AMP intentionally disabled "
    "for equivariant tensor products."
)


# ============================================================
# Read split manifest
# ============================================================

with open(
    SPLIT_FILE,
    "r"
) as f:

    split_info = json.load(f)


train_runs = split_info[
    "splits"
]["train"]

val_runs = split_info[
    "splits"
]["val"]

test_runs = split_info[
    "splits"
]["test"]


assert len(train_runs) == 12
assert len(val_runs) == 12
assert len(test_runs) == 12


print()
print(
    "Train trajectories:",
    len(train_runs)
)

print(
    "Val trajectories:  ",
    len(val_runs)
)

print(
    "Test trajectories: ",
    len(test_runs),
    "(manifest only)"
)


# ============================================================
# Verify TEST is physically absent
# ============================================================

test_present = []

for entry in test_runs:

    test_dir = os.path.join(
        ROOT,
        entry["run_id"]
    )

    if os.path.exists(
        test_dir
    ):

        test_present.append(
            entry["run_id"]
        )


if test_present:

    raise RuntimeError(
        "TEST DATA IS PRESENT IN RUNTIME:\n"
        + "\n".join(
            test_present
        )
    )


print(
    "Physical test trajectories present: 0"
)

print(
    "TEST SET IS SEALED."
)


# ============================================================
# Resolve sample paths
#
# Handles Windows backslashes in splits.json.
# ============================================================

def get_sample_paths(entries):

    paths = []

    for entry in entries:

        relative_sample_dir = (
            entry["sample_dir"]
            .replace("\\", "/")
        )

        sample_dir = os.path.join(
            ROOT,
            relative_sample_dir
        )

        if not os.path.isdir(
            sample_dir
        ):

            raise FileNotFoundError(
                f"{entry['run_id']}: "
                f"sample directory not found:\n"
                f"{sample_dir}"
            )

        files = sorted(
            os.path.join(
                sample_dir,
                f
            )
            for f in os.listdir(
                sample_dir
            )
            if (
                f.startswith(
                    "sample_"
                )
                and f.endswith(
                    ".npz"
                )
            )
        )

        expected = entry[
            "num_samples"
        ]

        if len(files) != expected:

            raise RuntimeError(
                f"{entry['run_id']}: "
                f"expected {expected}, "
                f"got {len(files)}"
            )

        paths.extend(
            files
        )

    return paths


train_paths = get_sample_paths(
    train_runs
)

val_paths = get_sample_paths(
    val_runs
)


assert len(train_paths) == 6000
assert len(val_paths) == 6000


print()
print(
    "Train samples:",
    len(train_paths)
)

print(
    "Val samples:  ",
    len(val_paths)
)

print(
    "First train sample:",
    train_paths[0]
)


# ============================================================
# Cached graph representation
# ============================================================

class CachedGraph:

    __slots__ = (
        "species",
        "velocity",
        "force",
        "edge_index",
        "edge_vector",
        "edge_distance",
        "y"
    )

    def __init__(
        self,
        species,
        velocity,
        force,
        edge_index,
        edge_vector,
        edge_distance,
        y
    ):

        self.species = species
        self.velocity = velocity
        self.force = force

        self.edge_index = edge_index

        self.edge_vector = edge_vector
        self.edge_distance = edge_distance

        self.y = y


# ============================================================
# Load TRAIN once
# + calculate TRAIN-ONLY normalization
# ============================================================

def load_train_cache(paths):

    cache = []

    vel_sum = np.zeros(
        3,
        dtype=np.float64
    )

    vel_sq_sum = np.zeros(
        3,
        dtype=np.float64
    )

    force_sum = np.zeros(
        3,
        dtype=np.float64
    )

    force_sq_sum = np.zeros(
        3,
        dtype=np.float64
    )

    n_atoms = 0

    start = time.time()

    for i, path in enumerate(
        paths
    ):

        with np.load(
            path
        ) as d:

            species = (
                d["atom_types"]
                .astype(
                    np.float32
                )[:, None]
            )

            velocity = np.stack(
                [
                    d["vx_t"],
                    d["vy_t"],
                    d["vz_t"]
                ],
                axis=1
            ).astype(
                np.float32
            )

            force = np.stack(
                [
                    d["fx_t"],
                    d["fy_t"],
                    d["fz_t"]
                ],
                axis=1
            ).astype(
                np.float32
            )

            src = d[
                "edge_src"
            ].astype(
                np.int64
            )

            dst = d[
                "edge_dst"
            ].astype(
                np.int64
            )

            edge_index = np.stack(
                [
                    src,
                    dst
                ],
                axis=0
            )

            edge_vector = (
                d["edge_vector"]
                .astype(
                    np.float32
                )
            )

            edge_distance = (
                d["edge_distance"]
                .astype(
                    np.float32
                )
            )

            y = (
                d[
                    "non_affine_displacement"
                ]
                .astype(
                    np.float32
                )
            )

        # --------------------------------------------
        # Train-only normalization statistics
        # --------------------------------------------

        vel64 = velocity.astype(
            np.float64
        )

        force64 = force.astype(
            np.float64
        )

        vel_sum += vel64.sum(
            axis=0
        )

        vel_sq_sum += (
            vel64 ** 2
        ).sum(
            axis=0
        )

        force_sum += force64.sum(
            axis=0
        )

        force_sq_sum += (
            force64 ** 2
        ).sum(
            axis=0
        )

        n_atoms += (
            velocity.shape[0]
        )

        cache.append(
            CachedGraph(
                torch.from_numpy(
                    species
                ),
                torch.from_numpy(
                    velocity
                ),
                torch.from_numpy(
                    force
                ),
                torch.from_numpy(
                    edge_index
                ),
                torch.from_numpy(
                    edge_vector
                ),
                torch.from_numpy(
                    edge_distance
                ),
                torch.from_numpy(
                    y
                )
            )
        )

        if (
            (i + 1) % 1000
            == 0
        ):

            print(
                f"Train cache: "
                f"{i+1}/{len(paths)}"
            )


    vel_mean = (
        vel_sum
        / n_atoms
    )

    force_mean = (
        force_sum
        / n_atoms
    )


    vel_var = (
        vel_sq_sum
        / n_atoms
        - vel_mean ** 2
    )

    force_var = (
        force_sq_sum
        / n_atoms
        - force_mean ** 2
    )


    vel_std = np.sqrt(
        np.maximum(
            vel_var,
            1e-12
        )
    )

    force_std = np.sqrt(
        np.maximum(
            force_var,
            1e-12
        )
    )


    norm = {

        "velocity_mean":
            torch.tensor(
                vel_mean,
                dtype=torch.float32
            ),

        "velocity_std":
            torch.tensor(
                vel_std,
                dtype=torch.float32
            ),

        "force_mean":
            torch.tensor(
                force_mean,
                dtype=torch.float32
            ),

        "force_std":
            torch.tensor(
                force_std,
                dtype=torch.float32
            )
    }


    print(
        f"Train cache loaded in "
        f"{time.time()-start:.1f}s"
    )

    return (
        cache,
        norm
    )


train_cache, norm = (
    load_train_cache(
        train_paths
    )
)


# ============================================================
# IMPORTANT EQUIVARIANCE NORMALIZATION
# ============================================================
#
# A vector representation must not be normalized with
# three substantially different axis-dependent scales if
# exact rotational equivariance is desired.
#
# The old NNConv experiment used per-component std.
#
# Here we convert those train statistics into ONE scalar
# normalization scale per vector field:
#
#   sigma_v = sqrt(mean(sigma_x^2, sigma_y^2, sigma_z^2))
#
#   sigma_f = sqrt(mean(...))
#
# Mean vector is also removed.
#
# This preserves vector transformation structure.
# ============================================================

velocity_scale = torch.sqrt(
    torch.mean(
        norm[
            "velocity_std"
        ] ** 2
    )
)

force_scale = torch.sqrt(
    torch.mean(
        norm[
            "force_std"
        ] ** 2
    )
)


print()
print(
    "TRAIN-ONLY NORMALIZATION"
)

print(
    "Velocity mean:",
    norm[
        "velocity_mean"
    ].numpy()
)

print(
    "Original component std:",
    norm[
        "velocity_std"
    ].numpy()
)

print(
    "Equivariant velocity scale:",
    velocity_scale.item()
)

print()

print(
    "Force mean:",
    norm[
        "force_mean"
    ].numpy()
)

print(
    "Original component std:",
    norm[
        "force_std"
    ].numpy()
)

print(
    "Equivariant force scale:",
    force_scale.item()
)


# ============================================================
# Normalize TRAIN cache once
#
# IMPORTANT:
# Mean subtraction technically selects a frame if the
# empirical mean is nonzero.
#
# The measured means are extremely small relative to the
# velocity scale. We preserve the train-centering used by
# the baseline while using isotropic scaling.
# ============================================================

for g in train_cache:

    g.velocity = (
        g.velocity
        - norm[
            "velocity_mean"
        ]
    ) / velocity_scale

    g.force = (
        g.force
        - norm[
            "force_mean"
        ]
    ) / force_scale


# ============================================================
# Load VAL once using TRAIN statistics
# ============================================================

def load_val_cache(
    paths,
    norm,
    velocity_scale,
    force_scale
):

    cache = []

    start = time.time()

    for i, path in enumerate(
        paths
    ):

        with np.load(
            path
        ) as d:

            species = (
                d["atom_types"]
                .astype(
                    np.float32
                )[:, None]
            )

            velocity = np.stack(
                [
                    d["vx_t"],
                    d["vy_t"],
                    d["vz_t"]
                ],
                axis=1
            ).astype(
                np.float32
            )

            force = np.stack(
                [
                    d["fx_t"],
                    d["fy_t"],
                    d["fz_t"]
                ],
                axis=1
            ).astype(
                np.float32
            )

            src = d[
                "edge_src"
            ].astype(
                np.int64
            )

            dst = d[
                "edge_dst"
            ].astype(
                np.int64
            )

            edge_index = np.stack(
                [
                    src,
                    dst
                ],
                axis=0
            )

            edge_vector = (
                d["edge_vector"]
                .astype(
                    np.float32
                )
            )

            edge_distance = (
                d["edge_distance"]
                .astype(
                    np.float32
                )
            )

            y = (
                d[
                    "non_affine_displacement"
                ]
                .astype(
                    np.float32
                )
            )


        velocity = (
            torch.from_numpy(
                velocity
            )
            - norm[
                "velocity_mean"
            ]
        ) / velocity_scale


        force = (
            torch.from_numpy(
                force
            )
            - norm[
                "force_mean"
            ]
        ) / force_scale


        cache.append(
            CachedGraph(
                torch.from_numpy(
                    species
                ),
                velocity,
                force,
                torch.from_numpy(
                    edge_index
                ),
                torch.from_numpy(
                    edge_vector
                ),
                torch.from_numpy(
                    edge_distance
                ),
                torch.from_numpy(
                    y
                )
            )
        )


        if (
            (i + 1) % 1000
            == 0
        ):

            print(
                f"Val cache: "
                f"{i+1}/{len(paths)}"
            )


    print(
        f"Val cache loaded in "
        f"{time.time()-start:.1f}s"
    )

    return cache


val_cache = load_val_cache(
    val_paths,
    norm,
    velocity_scale,
    force_scale
)


print()
print(
    "All 12,000 train+val "
    "graphs cached in RAM."
)


# ============================================================
# Zero predictor baseline
# ============================================================

def zero_baseline(
    cache
):

    squared = 0.0
    absolute = 0.0

    n = 0

    for g in cache:

        squared += torch.sum(
            g.y ** 2
        ).item()

        absolute += torch.sum(
            torch.abs(
                g.y
            )
        ).item()

        n += g.y.numel()

    mse = squared / n

    return (
        math.sqrt(
            mse
        ),
        absolute / n
    )


zero_rmse, zero_mae = (
    zero_baseline(
        val_cache
    )
)


print()
print(
    "VAL ZERO-PREDICTOR BASELINE"
)

print(
    f"RMSE: "
    f"{zero_rmse:.8f} Å"
)

print(
    f"MAE:  "
    f"{zero_mae:.8f} Å"
)


# ============================================================
# Dataset
# ============================================================

class EquivariantDataset(
    torch.utils.data.Dataset
):

    def __init__(
        self,
        cache,
        variant
    ):

        assert variant in [
            "A",
            "B",
            "C"
        ]

        self.cache = cache
        self.variant = variant


    def __len__(
        self
    ):

        return len(
            self.cache
        )


    def __getitem__(self, idx):

        g = self.cache[idx]

        return Data(
            species=g.species,
            velocity=g.velocity,
            force=g.force,
            edge_index=g.edge_index,
            edge_vector=g.edge_vector,
            edge_distance=g.edge_distance,
            y=g.y,
            num_nodes=g.species.shape[0]
        )


# ============================================================
# Irreps for each representation
# ============================================================

def input_irreps_for_variant(
    variant
):

    if variant == "A":

        return o3.Irreps(
            "1x0e"
        )

    elif variant == "B":

        return o3.Irreps(
            "1x0e + 1x1o"
        )

    else:

        return o3.Irreps(
            "1x0e + 2x1o"
        )


def build_input_features(
    data,
    variant
):

    if variant == "A":

        return data.species

    elif variant == "B":

        return torch.cat(
            [
                data.species,
                data.velocity
            ],
            dim=-1
        )

    else:

        return torch.cat(
            [
                data.species,
                data.velocity,
                data.force
            ],
            dim=-1
        )


# ============================================================
# Smooth radial basis
# ============================================================

def radial_basis(
    distance,
    cutoff=CUTOFF,
    num_basis=NUM_RADIAL
):

    # Normalize distance to [0,1].
    x = (
        distance
        / cutoff
    )

    centers = torch.linspace(
        0.0,
        1.0,
        num_basis,
        device=distance.device,
        dtype=distance.dtype
    )

    width = (
        1.0
        / num_basis
    )

    basis = torch.exp(
        -0.5
        * (
            (
                x[:, None]
                - centers[None, :]
            )
            / width
        ) ** 2
    )

    # Cosine cutoff envelope.
    envelope = (
        0.5
        * (
            torch.cos(
                math.pi
                * torch.clamp(
                    x,
                    0.0,
                    1.0
                )
            )
            + 1.0
        )
    )

    envelope = (
        envelope
        * (
            x <= 1.0
        ).to(
            distance.dtype
        )
    )

    return (
        basis
        * envelope[:, None]
    )


# ============================================================
# Equivariant message-passing layer
# ============================================================

class EquivariantConv(
    nn.Module
):

    def __init__(
        self,
        irreps_in,
        irreps_out
    ):

        super().__init__()

        self.irreps_in = o3.Irreps(
            irreps_in
        )

        self.irreps_out = o3.Irreps(
            irreps_out
        )

        self.irreps_edge = (
            EDGE_IRREPS
        )

        # ----------------------------------------------------
        # FullyConnectedTensorProduct determines all allowed
        # equivariant couplings:
        #
        # node irreps × spherical harmonics
        #       -> output irreps
        #
        # Internal weights are supplied per edge by radial MLP.
        # ----------------------------------------------------

        self.tp = (
            o3.FullyConnectedTensorProduct(
                self.irreps_in,
                self.irreps_edge,
                self.irreps_out,
                shared_weights=False,
                internal_weights=False
            )
        )

        self.radial_mlp = (
            nn.Sequential(

                nn.Linear(
                    NUM_RADIAL,
                    RADIAL_HIDDEN
                ),

                nn.SiLU(),

                nn.Linear(
                    RADIAL_HIDDEN,
                    self.tp.weight_numel
                )
            )
        )


    def forward(
        self,
        x,
        edge_index,
        edge_vector,
        edge_distance
    ):

        src = edge_index[0]
        dst = edge_index[1]

        # ----------------------------------------------------
        # Spherical harmonics of edge directions.
        #
        # normalize=True means direction magnitude does not
        # enter angular features.
        #
        # Distances separately enter radial network.
        # ----------------------------------------------------

        edge_sh = o3.spherical_harmonics(
            self.irreps_edge,
            edge_vector,
            normalize=True,
            normalization="component"
        )

        radial = radial_basis(
            edge_distance
        )

        weights = self.radial_mlp(
            radial
        )

        messages = self.tp(
            x[src],
            edge_sh,
            weights
        )

        # Mean aggregation, matching NNConv experiment.
        out = torch.zeros(
            (
                x.shape[0],
                self.irreps_out.dim
            ),
            dtype=x.dtype,
            device=x.device
        )

        out.index_add_(
            0,
            dst,
            messages
        )

        degree = torch.zeros(
            x.shape[0],
            dtype=x.dtype,
            device=x.device
        )

        degree.index_add_(
            0,
            dst,
            torch.ones(
                dst.shape[0],
                dtype=x.dtype,
                device=x.device
            )
        )

        out = (
            out
            / degree.clamp(
                min=1.0
            )[:, None]
        )

        return out


# ============================================================
# Equivariant nonlinearity
#
# Scalars can pass through SiLU.
#
# Vectors cannot have componentwise SiLU because that would
# break equivariance.
#
# Instead:
#
#       v -> sigmoid(a ||v|| + b) v
#
# where the gate is a rotational scalar.
# ============================================================

class EquivariantActivation(
    nn.Module
):

    def __init__(
        self,
        irreps
    ):

        super().__init__()

        self.irreps = o3.Irreps(
            irreps
        )

        self.vector_gates = (
            nn.ParameterList()
        )

        self.vector_biases = (
            nn.ParameterList()
        )

        for mul, ir in self.irreps:

            if ir.l > 0:

                self.vector_gates.append(
                    nn.Parameter(
                        torch.ones(
                            mul
                        )
                    )
                )

                self.vector_biases.append(
                    nn.Parameter(
                        torch.zeros(
                            mul
                        )
                    )
                )


    def forward(
        self,
        x
    ):

        pieces = []

        offset = 0
        vector_index = 0

        for mul, ir in self.irreps:

            block_dim = (
                mul
                * ir.dim
            )

            block = x[
                :,
                offset:
                offset + block_dim
            ]

            block = block.reshape(
                x.shape[0],
                mul,
                ir.dim
            )

            if ir.l == 0:

                block = F.silu(
                    block
                )

            else:

                norms = torch.sqrt(
                    torch.sum(
                        block ** 2,
                        dim=-1
                    )
                    + 1e-8
                )

                gate = torch.sigmoid(
                    norms
                    * self.vector_gates[
                        vector_index
                    ][None, :]
                    + self.vector_biases[
                        vector_index
                    ][None, :]
                )

                block = (
                    block
                    * gate[:, :, None]
                )

                vector_index += 1

            pieces.append(
                block.reshape(
                    x.shape[0],
                    block_dim
                )
            )

            offset += block_dim

        return torch.cat(
            pieces,
            dim=-1
        )


# ============================================================
# Full E(3)-equivariant dynamics model
# ============================================================

class E3DynamicsGNN(
    nn.Module
):

    def __init__(
        self,
        variant
    ):

        super().__init__()

        self.variant = variant

        self.irreps_input = (
            input_irreps_for_variant(
                variant
            )
        )

        self.irreps_hidden = (
            HIDDEN_IRREPS
        )

        self.irreps_output = (
            OUTPUT_IRREPS
        )

        # Equivariant linear embedding.
        self.input_linear = o3.Linear(
            self.irreps_input,
            self.irreps_hidden
        )

        self.conv1 = EquivariantConv(
            self.irreps_hidden,
            self.irreps_hidden
        )

        self.act1 = EquivariantActivation(
            self.irreps_hidden
        )

        self.conv2 = EquivariantConv(
            self.irreps_hidden,
            self.irreps_hidden
        )

        self.act2 = EquivariantActivation(
            self.irreps_hidden
        )

        self.output_linear = o3.Linear(
            self.irreps_hidden,
            self.irreps_output
        )


    def forward(
        self,
        data
    ):

        x = build_input_features(
            data,
            self.variant
        )

        x = self.input_linear(
            x
        )

        residual = x

        x = self.conv1(
            x,
            data.edge_index,
            data.edge_vector,
            data.edge_distance
        )

        x = self.act1(
            x
        )

        x = self.conv2(
            x,
            data.edge_index,
            data.edge_vector,
            data.edge_distance
        )

        x = (
            x
            + residual
        )

        x = self.act2(
            x
        )

        return self.output_linear(
            x
        )


# ============================================================
# Evaluation
# ============================================================

@torch.no_grad()
def evaluate(
    model,
    loader
):

    model.eval()

    sum_squared = 0.0
    sum_absolute = 0.0

    n_values = 0

    for batch in loader:

        batch = batch.to(
            DEVICE,
            non_blocking=True
        )

        pred = model(
            batch
        )

        error = (
            pred.float()
            - batch.y.float()
        )

        sum_squared += torch.sum(
            error ** 2
        ).item()

        sum_absolute += torch.sum(
            torch.abs(
                error
            )
        ).item()

        n_values += (
            error.numel()
        )

    mse = (
        sum_squared
        / n_values
    )

    return (
        mse,
        math.sqrt(
            mse
        ),
        sum_absolute
        / n_values
    )


# ============================================================
# Train one representation
# ============================================================

def train_variant(
    variant
):

    print()
    print(
        "=" * 72
    )

    print(
        f"E3 VARIANT {variant}"
    )

    print(
        "=" * 72
    )


    random.seed(
        SEED
    )

    np.random.seed(
        SEED
    )

    torch.manual_seed(
        SEED
    )

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            SEED
        )


    train_dataset = (
        EquivariantDataset(
            train_cache,
            variant
        )
    )

    val_dataset = (
        EquivariantDataset(
            val_cache,
            variant
        )
    )


    generator = (
        torch.Generator()
    )

    generator.manual_seed(
        SEED
    )


    train_loader = DataLoader(

        train_dataset,

        batch_size=BATCH_SIZE,

        shuffle=True,

        generator=generator,

        num_workers=0,

        pin_memory=(
            DEVICE.type
            == "cuda"
        )
    )


    val_loader = DataLoader(

        val_dataset,

        batch_size=BATCH_SIZE,

        shuffle=False,

        num_workers=0,

        pin_memory=(
            DEVICE.type
            == "cuda"
        )
    )


    model = E3DynamicsGNN(
        variant
    ).to(
        DEVICE
    )


    parameter_count = sum(
        p.numel()
        for p in model.parameters()
    )


    print(
        "Input irreps:",
        model.irreps_input
    )

    print(
        "Hidden irreps:",
        model.irreps_hidden
    )

    print(
        "Output irreps:",
        model.irreps_output
    )

    print(
        "Parameters:",
        parameter_count
    )


    optimizer = torch.optim.Adam(

        model.parameters(),

        lr=LR,

        weight_decay=WEIGHT_DECAY
    )


    criterion = nn.MSELoss()


    best_val = float(
        "inf"
    )

    best_epoch = 0

    evals_without_improvement = 0


    checkpoint = os.path.join(

        OUTPUT_DIR,

        f"best_e3_variant_{variant}.pt"
    )


    history = []

    variant_start = time.time()


    for epoch in range(
        1,
        EPOCHS + 1
    ):

        epoch_start = time.time()

        model.train()

        train_squared = 0.0
        train_values = 0


        for batch in train_loader:

            batch = batch.to(
                DEVICE,
                non_blocking=True
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            pred = model(
                batch
            )

            loss = criterion(
                pred,
                batch.y
            )

            if not torch.isfinite(
                loss
            ):

                raise RuntimeError(
                    f"Non-finite loss in "
                    f"variant {variant}, "
                    f"epoch {epoch}"
                )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=10.0
            )

            optimizer.step()


            error = (
                pred.detach()
                - batch.y
            )

            train_squared += torch.sum(
                error ** 2
            ).item()

            train_values += (
                error.numel()
            )


        train_mse = (
            train_squared
            / train_values
        )

        train_rmse = math.sqrt(
            train_mse
        )


        do_validation = (

            epoch == 1

            or epoch
            % VAL_EVERY
            == 0

            or epoch
            == EPOCHS
        )


        val_mse = None
        val_rmse = None
        val_mae = None


        if do_validation:

            (
                val_mse,
                val_rmse,
                val_mae
            ) = evaluate(
                model,
                val_loader
            )


            if val_mse < best_val:

                best_val = val_mse
                best_epoch = epoch

                evals_without_improvement = 0

                torch.save(
                    model.state_dict(),
                    checkpoint
                )

            else:

                evals_without_improvement += 1


        history.append(
            {
                "epoch":
                    epoch,

                "train_rmse":
                    float(
                        train_rmse
                    ),

                "val_rmse":
                    None
                    if val_rmse
                    is None
                    else float(
                        val_rmse
                    ),

                "val_mae":
                    None
                    if val_mae
                    is None
                    else float(
                        val_mae
                    )
            }
        )


        if do_validation:

            print(

                f"Epoch {epoch:03d} | "

                f"train RMSE "
                f"{train_rmse:.6e} Å | "

                f"val RMSE "
                f"{val_rmse:.6e} Å | "

                f"val MAE "
                f"{val_mae:.6e} Å | "

                f"{time.time()-epoch_start:.1f}s"
            )


        if (
            do_validation
            and
            evals_without_improvement
            >= PATIENCE_EVALS
        ):

            print(
                f"Early stopping at "
                f"epoch {epoch}. "
                f"Best epoch: "
                f"{best_epoch}"
            )

            break


    # ========================================================
    # Reload exact best checkpoint
    # ========================================================

    model.load_state_dict(

        torch.load(

            checkpoint,

            map_location=DEVICE,

            weights_only=True
        )
    )


    (
        val_mse,
        val_rmse,
        val_mae
    ) = evaluate(
        model,
        val_loader
    )


    elapsed = (
        time.time()
        - variant_start
    )


    print()
    print(
        f"BEST E3 VARIANT {variant}"
    )

    print(
        "Best epoch:",
        best_epoch
    )

    print(
        f"Validation RMSE: "
        f"{val_rmse:.8f} Å"
    )

    print(
        f"Validation MAE:  "
        f"{val_mae:.8f} Å"
    )

    print(
        f"Wall time: "
        f"{elapsed/60:.2f} min"
    )


    with open(

        os.path.join(
            OUTPUT_DIR,
            f"history_e3_variant_{variant}.json"
        ),

        "w"

    ) as f:

        json.dump(
            history,
            f,
            indent=2
        )


    del model
    del optimizer

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    return {

        "variant":
            variant,

        "best_epoch":
            best_epoch,

        "val_mse":
            float(
                val_mse
            ),

        "val_rmse":
            float(
                val_rmse
            ),

        "val_mae":
            float(
                val_mae
            ),

        "parameters":
            parameter_count,

        "wall_seconds":
            float(
                elapsed
            ),

        "checkpoint":
            checkpoint
    }


# ============================================================
# Train E3 A/B/C
# ============================================================

experiment_start = time.time()

results = []

for variant in [
    "A",
    "B",
    "C"
]:

    results.append(
        train_variant(
            variant
        )
    )


# ============================================================
# A/B/C validation comparison
# ============================================================

print()
print(
    "=" * 72
)

print(
    "E3 A/B/C VALIDATION COMPARISON"
)

print(
    "=" * 72
)

print()

print(
    f"Zero baseline | "
    f"RMSE {zero_rmse:.8f} Å | "
    f"MAE {zero_mae:.8f} Å"
)


for r in results:

    improvement = (
        1.0
        - r["val_rmse"]
        / zero_rmse
    ) * 100.0

    print(

        f"{r['variant']} | "

        f"epoch "
        f"{r['best_epoch']:3d} | "

        f"RMSE "
        f"{r['val_rmse']:.8f} Å | "

        f"MAE "
        f"{r['val_mae']:.8f} Å | "

        f"vs-zero "
        f"{improvement:+.2f}%"
    )


# ============================================================
# Random SO(3) rotations
# ============================================================

def random_rotation_matrix(
    rng
):

    A = rng.normal(
        size=(
            3,
            3
        )
    )

    Q, R = np.linalg.qr(
        A
    )

    signs = np.sign(
        np.diag(
            R
        )
    )

    signs[
        signs == 0
    ] = 1.0

    Q = (
        Q
        @ np.diag(
            signs
        )
    )

    if np.linalg.det(
        Q
    ) < 0:

        Q[:, 0] *= -1.0

    return Q.astype(
        np.float32
    )


rng = np.random.default_rng(
    ROTATION_SEED
)

rotations = [

    random_rotation_matrix(
        rng
    )

    for _ in range(
        N_ROTATIONS
    )
]


print()
print(
    f"Generated "
    f"{N_ROTATIONS} "
    f"SO(3) rotations."
)


# ============================================================
# Rotated dataset
#
# IMPORTANT:
#
# val_cache currently contains normalized velocity/force.
#
# Because normalization is isotropic:
#
#       v_norm' = v_norm @ R.T
#
# except for the tiny empirical mean subtraction.
#
# For a strict rotation test, reconstruct physical vectors,
# rotate them, then normalize using the correspondingly
# rotated training mean.
#
# This ensures the preprocessing itself is equivariant.
# ============================================================

class RotatedEquivariantDataset(
    torch.utils.data.Dataset
):

    def __init__(
        self,
        cache,
        variant,
        R
    ):

        self.cache = cache
        self.variant = variant

        self.R = torch.tensor(
            R,
            dtype=torch.float32
        )


    def __len__(
        self
    ):

        return len(
            self.cache
        )


    def __getitem__(
        self,
        idx
    ):

        g = self.cache[idx]

        RT = self.R.T

        # Since g.velocity and g.force are already
        # isotropically normalized vectors, rotating
        # them directly is mathematically equivariant.

        velocity = (
            g.velocity
            @ RT
        )

        force = (
            g.force
            @ RT
        )

        edge_vector = (
            g.edge_vector
            @ RT
        )

        target = (
            g.y
            @ RT
        )


        return Data(
            species=g.species,
            velocity=velocity,
            force=force,
            edge_index=g.edge_index,
            edge_vector=edge_vector,
            edge_distance=g.edge_distance,
            y=target,
            num_nodes=g.species.shape[0]
        )

# ============================================================
# Prediction helper
# ============================================================

@torch.no_grad()
def predict_dataset(
    model,
    dataset
):

    loader = DataLoader(

        dataset,

        batch_size=BATCH_SIZE,

        shuffle=False,

        num_workers=0,

        pin_memory=(
            DEVICE.type
            == "cuda"
        )
    )


    predictions = []
    targets = []


    for batch in loader:

        batch = batch.to(
            DEVICE,
            non_blocking=True
        )

        pred = model(
            batch
        )

        predictions.append(
            pred.float().cpu()
        )

        targets.append(
            batch.y.float().cpu()
        )


    return (
        torch.cat(
            predictions,
            dim=0
        ),

        torch.cat(
            targets,
            dim=0
        )
    )


def metric_dict(
    pred,
    target
):

    error = (
        pred
        - target
    )

    mse = torch.mean(
        error ** 2
    ).item()

    rmse = math.sqrt(
        mse
    )

    mae = torch.mean(
        torch.abs(
            error
        )
    ).item()


    return {
        "rmse":
            float(
                rmse
            ),

        "mae":
            float(
                mae
            )
    }


# ============================================================
# Rotation test for one variant
# ============================================================

def rotation_test_variant(
    result
):

    variant = result[
        "variant"
    ]

    print()
    print(
        "=" * 72
    )

    print(
        f"E3 ROTATION TEST — "
        f"VARIANT {variant}"
    )

    print(
        "=" * 72
    )


    model = E3DynamicsGNN(
        variant
    ).to(
        DEVICE
    )


    model.load_state_dict(

        torch.load(

            result[
                "checkpoint"
            ],

            map_location=DEVICE,

            weights_only=True
        )
    )


    model.eval()


    original_dataset = (
        EquivariantDataset(
            val_cache,
            variant
        )
    )


    (
        original_pred,
        original_target
    ) = predict_dataset(
        model,
        original_dataset
    )


    original_metrics = metric_dict(
        original_pred,
        original_target
    )


    print(
        f"Original RMSE: "
        f"{original_metrics['rmse']:.8f} Å"
    )


    # --------------------------------------------
    # Ensure checkpoint reproduction
    # --------------------------------------------

    difference = abs(
        original_metrics[
            "rmse"
        ]
        - result[
            "val_rmse"
        ]
    )


    print(
        f"Checkpoint difference: "
        f"{difference:.3e} Å"
    )


    if difference > 5e-6:

        raise RuntimeError(
            f"Variant {variant} checkpoint "
            f"failed reproduction gate."
        )


    rotation_results = []


    for rotation_id, R in enumerate(
        rotations,
        start=1
    ):

        rotated_dataset = (
            RotatedEquivariantDataset(
                val_cache,
                variant,
                R
            )
        )


        (
            rotated_pred,
            rotated_target
        ) = predict_dataset(
            model,
            rotated_dataset
        )


        accuracy = metric_dict(
            rotated_pred,
            rotated_target
        )


        R_tensor = torch.tensor(
            R,
            dtype=torch.float32
        )


        expected_rotated_pred = (
            original_pred
            @ R_tensor.T
        )


        equivariance = metric_dict(
            rotated_pred,
            expected_rotated_pred
        )


        degradation = (
            accuracy[
                "rmse"
            ]
            / original_metrics[
                "rmse"
            ]
        )


        rotation_results.append(
            {
                "rotation_id":
                    rotation_id,

                "rotation_matrix":
                    R.tolist(),

                "determinant":
                    float(
                        np.linalg.det(
                            R
                        )
                    ),

                "rotated_target_rmse":
                    accuracy[
                        "rmse"
                    ],

                "rotated_target_mae":
                    accuracy[
                        "mae"
                    ],

                "equivariance_rmse":
                    equivariance[
                        "rmse"
                    ],

                "equivariance_mae":
                    equivariance[
                        "mae"
                    ],

                "accuracy_degradation_factor":
                    float(
                        degradation
                    )
            }
        )


        print(

            f"Rotation "
            f"{rotation_id:02d} | "

            f"target RMSE "
            f"{accuracy['rmse']:.8f} Å | "

            f"equiv RMSE "
            f"{equivariance['rmse']:.8e} Å | "

            f"degradation "
            f"{degradation:.4f}x"
        )


    rotated_rmse = np.array(
        [
            r[
                "rotated_target_rmse"
            ]
            for r in rotation_results
        ]
    )


    equiv_rmse = np.array(
        [
            r[
                "equivariance_rmse"
            ]
            for r in rotation_results
        ]
    )


    degradation = (
        rotated_rmse
        / original_metrics[
            "rmse"
        ]
    )


    summary = {

        "variant":
            variant,

        "original_rmse":
            original_metrics[
                "rmse"
            ],

        "rotated_rmse_mean":
            float(
                rotated_rmse.mean()
            ),

        "rotated_rmse_std":
            float(
                rotated_rmse.std(
                    ddof=1
                )
            ),

        "equivariance_rmse_mean":
            float(
                equiv_rmse.mean()
            ),

        "equivariance_rmse_std":
            float(
                equiv_rmse.std(
                    ddof=1
                )
            ),

        "degradation_mean":
            float(
                degradation.mean()
            ),

        "degradation_min":
            float(
                degradation.min()
            ),

        "degradation_max":
            float(
                degradation.max()
            ),

        "rotations":
            rotation_results
    }


    print()
    print(
        f"VARIANT {variant} "
        f"ROTATION SUMMARY"
    )

    print(
        f"Original RMSE: "
        f"{original_metrics['rmse']:.8f} Å"
    )

    print(
        f"Rotated RMSE:  "
        f"{rotated_rmse.mean():.8f} "
        f"± "
        f"{rotated_rmse.std(ddof=1):.8f} Å"
    )

    print(
        f"Equiv RMSE:    "
        f"{equiv_rmse.mean():.8e} "
        f"± "
        f"{equiv_rmse.std(ddof=1):.8e} Å"
    )

    print(
        f"Degradation:   "
        f"{degradation.mean():.5f}x"
    )


    del model

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    return summary


# ============================================================
# Run rotation tests for A/B/C
# ============================================================

rotation_summaries = []

for result in results:

    rotation_summaries.append(
        rotation_test_variant(
            result
        )
    )


# ============================================================
# FINAL SUMMARY
# ============================================================

print()
print(
    "=" * 90
)

print(
    "FINAL E3NN A/B/C + ROTATION COMPARISON"
)

print(
    "=" * 90
)


print()
print(
    f"Zero predictor RMSE: "
    f"{zero_rmse:.8f} Å"
)

print()


for result, rotation in zip(
    results,
    rotation_summaries
):

    improvement = (
        1.0
        - result[
            "val_rmse"
        ]
        / zero_rmse
    ) * 100.0


    print(

        f"{result['variant']} | "

        f"val RMSE "
        f"{result['val_rmse']:.8f} Å | "

        f"vs-zero "
        f"{improvement:+.2f}% | "

        f"rot RMSE "
        f"{rotation['rotated_rmse_mean']:.8f} Å | "

        f"equiv RMSE "
        f"{rotation['equivariance_rmse_mean']:.3e} Å | "

        f"rot degradation "
        f"{rotation['degradation_mean']:.5f}x"
    )



total_minutes = (
    time.time()
    - experiment_start
) / 60.0


output = {

    "experiment":
        "e3nn_abc_plus_rotation",

    "torch_version":
        torch.__version__,

    "e3nn_version":
        e3nn.__version__,

    "seed":
        SEED,

    "rotation_seed":
        ROTATION_SEED,

    "n_rotations":
        N_ROTATIONS,

    "train_samples":
        len(train_paths),

    "val_samples":
        len(val_paths),

    "test_set_used":
        False,

    "zero_baseline_rmse":
        float(
            zero_rmse
        ),

    "zero_baseline_mae":
        float(
            zero_mae
        ),

    "hidden_irreps":
        str(
            HIDDEN_IRREPS
        ),

    "edge_irreps":
        str(
            EDGE_IRREPS
        ),

    "output_irreps":
        str(
            OUTPUT_IRREPS
        ),

    "results":
        results,

    "rotation_results":
        rotation_summaries,

    "total_minutes":
        float(
            total_minutes
        )
}


OUTPUT_FILE = os.path.join(
    OUTPUT_DIR,
    "e3nn_abc_rotation_results.json"
)


with open(
    OUTPUT_FILE,
    "w"
) as f:

    json.dump(
        output,
        f,
        indent=2
    )


print()
print(
    f"Total experiment time: "
    f"{total_minutes:.2f} min"
)

print(
    "Saved:",
    OUTPUT_FILE
)

print()

print(
    "TEST SET HAS NOT BEEN "
    "LOADED OR EVALUATED."
)

In [2]:
!nvidia-smi


Wed Sep 23 02:23:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----